# PetBreed AI training

Select a GPU runtime when available, then run the cells in order. Sign in to mount Google Drive. Each run saves its checkpoint, metrics, and environment details to a new Drive folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY_URL = 'https://github.com/nikalab7/Petdetect-.git'
PROJECT_DIR = Path('/content/Petdetect-')
if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)

subprocess.run(['python', '-m', 'pip', 'install', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT_DIR), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Device: {DEVICE}; source commit: {COMMIT}')

In [ ]:
import importlib.metadata
import json
import sys
from datetime import datetime, timezone

RUN_DIR = Path('/content/drive/MyDrive/PetBreedAI/runs') / datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_utc')
RUN_DIR.mkdir(parents=True, exist_ok=False)
DATA_DIR = Path('/content/PetBreedAI-data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
packages = ['torch', 'torchvision', 'scikit-learn', 'numpy', 'pillow', 'streamlit', 'pytest']
metadata = {
    'source_commit': COMMIT,
    'python_version': sys.version,
    'device': DEVICE,
    'cuda_version': torch.version.cuda,
    'packages': {name: importlib.metadata.version(name) for name in packages},
}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(f'Results will be saved in: {RUN_DIR}')

In [ ]:
command = [
    sys.executable, str(PROJECT_DIR / 'train.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(RUN_DIR),
    '--head-epochs', '3', '--tune-epochs', '3',
    '--batch-size', '32', '--seed', '42', '--device', 'auto',
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)

results = json.loads((RUN_DIR / 'train_results.json').read_text(encoding='utf-8'))
print(f"Selected stage: {results['selected_stage']}")
print(f"Best validation macro-F1: {results['best_validation_macro_f1']:.4f}")
print(f"Run folder: {RUN_DIR}")

## Evaluate the completed run

Run this cell once in the same Colab session after training has completed. It uses the existing dataset and run folder, measures latency on Colab CPU, and refuses to start the official test twice for the same run.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

project_dir = Path(PROJECT_DIR)
data_dir = Path(DATA_DIR)
run_dir = Path(RUN_DIR).resolve()
expected_run = Path('/content/drive/MyDrive/PetBreedAI/runs/20261026_155151_utc').resolve()
if run_dir != expected_run:
    raise RuntimeError(f'Unexpected run folder: {run_dir}')
if not (run_dir / 'best.pt').is_file() or not (run_dir / 'train_results.json').is_file():
    raise FileNotFoundError(f'Training files are missing from {run_dir}')
if not data_dir.is_dir():
    raise FileNotFoundError(f'The existing Colab dataset folder is missing: {data_dir}')

training = json.loads((run_dir / 'train_results.json').read_text(encoding='utf-8'))
print(f"Training run: {run_dir.name}; selected stage: {training['selected_stage']}")
print(f"Best validation macro-F1: {training['best_validation_macro_f1']:.4f}")
subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True)
command = [
    sys.executable, str(project_dir / 'evaluate.py'),
    '--data-dir', str(data_dir), '--out-dir', str(run_dir),
    '--batch-size', '32', '--device', 'cpu', '--no-download',
]
subprocess.run(command, cwd=project_dir, check=True)

evaluation = json.loads((run_dir / 'eval_results.json').read_text(encoding='utf-8'))
print(f"Selected input size: {evaluation['optimization']['selected_image_size']}px")
print(f"Held-out test: {evaluation['held_out_test']}")
print(evaluation['benchmark_note'])
print(f"Saved results in {run_dir}")

## View the completed evaluation outputs

This cell only reads the completed run. It does not load a model for inference, rerun evaluation, or download the dataset. It plots the saved confusion matrix and displays the first five saved test errors.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision.datasets import OxfordIIITPet

run_dir = Path(RUN_DIR).resolve()
data_dir = Path(DATA_DIR)
expected_run = Path('/content/drive/MyDrive/PetBreedAI/runs/20261026_155151_utc').resolve()
if run_dir != expected_run:
    raise RuntimeError(f'Unexpected run folder: {run_dir}')

results_path = run_dir / 'eval_results.json'
matrix_path = run_dir / 'confusion_matrix.npy'
checkpoint_path = run_dir / 'best.pt'
for path in (results_path, matrix_path, checkpoint_path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing completed evaluation file: {path}')

results = json.loads(results_path.read_text(encoding='utf-8'))
comparison = results['optimization']['validation_resolution_comparison']
for size in ('224', '160'):
    row = comparison[size]
    print(f"Validation {size}px: Accuracy={row['accuracy']:.5f}, "
          f"Macro-F1={row['macro_f1']:.5f}, "
          f"median latency={row['median_latency_ms']:.2f} ms ({row['device']})")
print(f"Selected resolution: {results['optimization']['selected_image_size']}px")
print(f"Held-out test: {results['held_out_test']}")
print(results['benchmark_note'])

matrix = np.load(matrix_path)
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
class_names = checkpoint['class_names']
if matrix.shape != (len(class_names), len(class_names)):
    raise ValueError(f'Unexpected confusion-matrix shape: {matrix.shape}')
row_totals = matrix.sum(axis=1, keepdims=True)
normalized = np.divide(matrix, row_totals, out=np.zeros_like(matrix, dtype=float), where=row_totals != 0)
fig, ax = plt.subplots(figsize=(18, 16))
image = ax.imshow(normalized, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax.set_title('Official test confusion matrix (row-normalized)')
ax.set_xlabel('Predicted breed')
ax.set_ylabel('True breed')
ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=90, fontsize=7)
ax.set_yticklabels(class_names, fontsize=7)
fig.colorbar(image, ax=ax, label='Fraction of each true class')
fig.tight_layout()
confusion_image = run_dir / 'confusion_matrix.png'
fig.savefig(confusion_image, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {confusion_image}')

errors = results.get('errors_sample', [])[:5]
if len(errors) < 5:
    raise ValueError(f'Expected five saved mistakes, found {len(errors)}')
if not data_dir.is_dir():
    raise FileNotFoundError(f'The existing Colab dataset folder is missing: {data_dir}')
test_data = OxfordIIITPet(root=str(data_dir), split='test', target_types='category', download=False)
if list(test_data.classes) != list(class_names):
    raise ValueError('Test dataset class order does not match the saved checkpoint')
examples_dir = run_dir / 'misclassified_examples'
examples_dir.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, item in zip(axes, errors):
    index = int(item['dataset_index'])
    photo, true_label = test_data[index]
    actual_name = test_data.classes[int(true_label)]
    if actual_name != item['expected']:
        raise ValueError(f'Label mismatch at test index {index}: {actual_name} != {item["expected"]}')
    photo = photo.convert('RGB')
    photo.save(examples_dir / f'{index:04d}.jpg', quality=92)
    ax.imshow(photo)
    ax.set_title(f"Test #{index}\nTrue: {item['expected']}\nPredicted: {item['predicted']}", fontsize=9)
    ax.axis('off')
fig.tight_layout()
examples_image = run_dir / 'misclassified_examples.png'
fig.savefig(examples_image, dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved the five original test photos in:', examples_dir)
print('Saved montage:', examples_image)

## Download the completed run for local use

Run after the inspection cell. This creates one ZIP outside the Drive run folder and downloads the existing checkpoint, metrics, confusion-matrix files, and five displayed test images. It does not train or evaluate the model.

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

run_dir = Path(RUN_DIR).resolve()
expected_run = Path('/content/drive/MyDrive/PetBreedAI/runs/20261026_155151_utc').resolve()
if run_dir != expected_run:
    raise RuntimeError(f'Unexpected run folder: {run_dir}')
required = [
    'best.pt', 'train_results.json', 'eval_results.json',
    'confusion_matrix.npy', 'confusion_matrix.png', 'misclassified_examples.png',
]
paths = [run_dir / name for name in required]
examples_dir = run_dir / 'misclassified_examples'
example_paths = sorted(examples_dir.glob('*.jpg'))
if len(example_paths) < 5:
    raise FileNotFoundError(f'Expected five saved example photos in {examples_dir}')
missing = [str(path) for path in paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing run outputs: ' + ', '.join(missing))
metadata = run_dir / 'run_metadata.json'
if metadata.is_file():
    paths.append(metadata)
paths.extend(example_paths[:5])

archive = Path('/content/PetBreedAI_20261026_results.zip')
with ZipFile(archive, 'w', compression=ZIP_DEFLATED) as zip_file:
    for path in paths:
        zip_file.write(path, arcname=str(path.relative_to(run_dir)))
print(f'Created {archive} with {len(paths)} files')
files.download(str(archive))